In [ ]:
import torch
if torch.cuda.is_available():
    print("gpu available")
    print(torch.cuda.get_device_name(0))
else:
    print("no gpu available")

gpu available
NVIDIA A100-SXM4-40GB


In [ ]:
!pip install wfdb==4.1.2 pandas==2.2.2 numpy==1.26.4 --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.0/160.0 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 162.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import wfdb #load ecg data from physionet
import numpy as np # numerical operations
import matplotlib.pyplot as plt # plotting
import torch # build model
import torch.nn as nn # neural network, defines layers and structure of model
import torch.optim as optim # optimizers used to adjust model weights
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader # batch data during training + eval
from sklearn.model_selection import train_test_split # to split and evaluate data
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from google.colab import drive
import gc
import copy
import os
from datetime import date, datetime
from zoneinfo import ZoneInfo
import pickle

In [ ]:
drive.mount('/content/drive')
# alter save_path to match where you downloaded your ecg_data.npz!
save_path = '/content/drive/MyDrive/ZiyaAhmad/12sec_ecg_data.npz'

data = np.load(save_path)
all_windows = data['windows']
all_labels_3class = data['labels']

# alter split_path to match where you downloaded your train_test_split!
split_path = '/content/drive/MyDrive/ZiyaAhmad/12sec_train_test_split.npz'
split_data = np.load(split_path)
train_indices = split_data['train_indices']
test_indices = split_data['test_indices']

print(f"loaded train/test split!")
print(f"train samples: {len(train_indices):,}")
print(f"test samples: {len(test_indices):,}")

Mounted at /content/drive
loaded train/test split!
train samples: 197,044
test samples: 49,262


In [ ]:
def augment_presva_samples(windows,labels,indices,aug_factor=2):
    augmented_windows = []
    augmented_labels = []
    augmented_indices_map = []

    for idx in indices:
        window = windows[idx]
        label = labels[idx]

        augmented_windows.append(window)
        augmented_labels.append(label)
        augmented_indices_map.append(idx)

        if label == 2:
            original = window.copy()

            for aug_num in range(aug_factor):
                aug_type = ['noise','scale','combo'][aug_num % 3]

                if aug_type == 'noise':
                    noise_level = np.random.uniform(0.008,0.018)
                    augmented = original + np.random.normal(0,noise_level,original.shape)

                elif aug_type == 'scale':
                    scale_factor = np.random.uniform(0.92,1.08)
                    augmented = original*scale_factor

                elif aug_type == 'combo':
                    noise_level = np.random.uniform(0.005,0.012)
                    scale_factor = np.random.uniform(0.94,1.06)
                    shift_amount = np.random.randint(-10,10)

                    augmented = np.roll(original,shift_amount)
                    augmented = augmented*scale_factor
                    augmented = augmented + np.random.normal(0,noise_level,augmented.shape)


                augmented_windows.append(augmented)
                augmented_labels.append(2)
                augmented_indices_map.append(idx)

    return np.array(augmented_windows),np.array(augmented_labels)

train_labels_original = all_labels_3class[train_indices]
print(f"original training set class distribution:")
print(f"  Normal: {np.sum(train_labels_original == 0):,}")
print(f"  SVA: {np.sum(train_labels_original == 1):,}")
print(f"  Pre-SVA: {np.sum(train_labels_original == 2):,}")
print(f"  Total: {len(train_labels_original):,}")
presva_count_orig = np.sum(train_labels_original == 2)
sva_count_orig = np.sum(train_labels_original == 1)
print(f"  Ratio SVA:Pre-SVA = {sva_count_orig/presva_count_orig:.2f}:1")

train_windows_aug, train_labels_aug = augment_presva_samples(all_windows,all_labels_3class,train_indices,aug_factor=3)

print(f"\naugmented training set class distribution:")
print(f"  Normal: {np.sum(train_labels_aug == 0):,}")
print(f"  SVA: {np.sum(train_labels_aug == 1):,}")
print(f"  Pre-SVA: {np.sum(train_labels_aug == 2):,}")
print(f"  Total: {len(train_labels_aug):,}")
presva_count_aug = np.sum(train_labels_aug == 2)
sva_count_aug = np.sum(train_labels_aug == 1)
print(f"  Ratio SVA:Pre-SVA = {sva_count_aug/presva_count_aug:.2f}:1")

test_windows = all_windows[test_indices]
test_labels_3class = all_labels_3class[test_indices]

original training set class distribution:
  Normal: 116,394
  SVA: 62,565
  Pre-SVA: 18,085
  Total: 197,044
  Ratio SVA:Pre-SVA = 3.46:1

augmented training set class distribution:
  Normal: 116,394
  SVA: 62,565
  Pre-SVA: 72,340
  Total: 251,299
  Ratio SVA:Pre-SVA = 0.86:1


In [ ]:
# DO NOT run this kernel 2x in the same session (will result in double labelling)

# convert to binary labels: normal=0, abnormal(sva or pre-sva)=1
train_labels_binary = train_labels_aug.copy()
train_labels_binary[train_labels_binary == 2] = 1  # pre-sva → 1
train_labels_binary[train_labels_binary == 1] = 1  # sva → 1

test_labels_binary = test_labels_3class.copy()
test_labels_binary[test_labels_binary == 2] = 1
test_labels_binary[test_labels_binary == 1] = 1

# normalize augmented training data
scaler = StandardScaler()
train_windows_flat = train_windows_aug.reshape(-1, 1)
train_windows_scaled = scaler.fit_transform(train_windows_flat).reshape(train_windows_aug.shape)
del train_windows_flat
gc.collect()

# normalize test data with same scaler (no augmentation on test!)
test_windows_flat = test_windows.reshape(-1, 1)
test_windows_scaled = scaler.transform(test_windows_flat).reshape(test_windows.shape)
del test_windows_flat
gc.collect()

# convert to tensors
X_train = torch.tensor(train_windows_scaled, dtype=torch.float32).unsqueeze(-1)
y_train = torch.tensor(train_labels_binary, dtype=torch.float32)
del train_windows_scaled
gc.collect()

X_test = torch.tensor(test_windows_scaled, dtype=torch.float32).unsqueeze(-1)
y_test = torch.tensor(test_labels_binary, dtype=torch.float32)
del test_windows_scaled
gc.collect()

y_test_3class = test_labels_3class  # keep for analysis

print(f"\nTest set breakdown:")
print(f"  Normal: {np.sum(y_test_3class == 0):,}")
print(f"  SVA: {np.sum(y_test_3class == 1):,}")
print(f"  Pre-SVA: {np.sum(y_test_3class == 2):,}")

print(f"\nFinal tensor shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  X_test: {X_test.shape}")
print(f"  y_test: {y_test.shape}")

# create dataloaders
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


Test set breakdown:
  Normal: 29,099
  SVA: 15,641
  Pre-SVA: 4,522

Final tensor shapes:
  X_train: torch.Size([251299, 1536, 1])
  y_train: torch.Size([251299])
  X_test: torch.Size([49262, 1536, 1])
  y_test: torch.Size([49262])


In [ ]:
class SelfAttention(nn.Module):
    def __init__(self,hidden_dim):
        super(SelfAttention,self).__init__()
        self.attention = nn.Linear(hidden_dim,1)

    def forward(self,gru_output): #gru_output_shape: (batch, seq_len, hidden_dim)
        # calc attention scores for ea/ time step
        attn_scores = self.attention(gru_output) # (batch, seq_len, 1)
        attn_weights = F.softmax(attn_scores,dim=1)

        # apply attention weights
        context = torch.sum(attn_weights * gru_output, dim=1)

        return context,attn_weights

class GRUWithAttention(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2, dropout=0.3):
        super(GRUWithAttention, self).__init__()

        # main gru
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        # self attention layer
        self.attention = SelfAttention(hidden_dim)

        # fully connected layers
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim,1)
        self.sigmoid = nn.Sigmoid() # using sigmoid w/ BCELoss

    def forward(self,x):
        # x shape: (batch, seq_len, input_dim)

        # gru forward pass (get outputs @ ALL time stamps)
        gru_out, _ = self.gru(x)

        # apply self-attention
        attended, attn_weights = self.attention(gru_out)

        # dropout
        attended = self.dropout(attended)

        # final classification
        out = self.fc(attended)

        return self.sigmoid(out) # apply sigmoid for BCELoss
        #return out # use this for BCEWithLogitsLoss


device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # check if theres gpu

model = GRUWithAttention(dropout=0.3).to(device) # creates model on device (cpu/gpu)
criterion = nn.BCELoss()  # Binary Cross-Entropy (BCE) for binary classification
optimizer = optim.Adam(model.parameters(), lr=0.0003) # adam optimizer to update model weights in training

print("Training samples: ", len(X_train))
print("Test samples: ", len(X_test))

Training samples:  251299
Test samples:  49262


In [ ]:
num_epochs = 300

# can alter early-stopping based on which metric you're trying to maximize
# (for example, the code below performs early stopping based on the macro-avg)
train_losses = []
test_losses = []
test_accuracies = []
macro_avgs = []

best_macro_avg = 0
best_accuracy = 0
best_model_state = None
best_abnormal_recall_state = None
patience = 0
max_patience = 20

best_abnormal_recall = 0
target_abnormal_recall = 0.89

for epoch in range(num_epochs):
    model.train() # put model in traning mode
    epoch_loss = 0 # accumulate loss for this epoch
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device) # move to gpu if available
        y_batch = y_batch.to(device).unsqueeze(1) # reshape: (32,) -> (32,1)

        outputs = model(X_batch) # forward pass, get model predictions

        loss = criterion(outputs, y_batch) # calculate loss

        # backward pass: calc gradients
        optimizer.zero_grad() #clear old gradients
        loss.backward() # calculate new gradients
        optimizer.step() # update model weights

        epoch_loss += loss.item() # track total loss for this epoch

        # calc training accuracy
        predicted = (outputs > 0.5).float()
        train_correct += (predicted == y_batch).sum().item()
        train_total += y_batch.size(0)

    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    train_acc = train_correct / train_total

    model.eval() # put model in evaluation mode
    test_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device) # y_batch here is (batch_size,)

            outputs_model = model(X_batch) # outputs_model is (batch_size, 1)

            #convert probabilities to predictions
            # if output > 0.5 -> predict 1 (abnormal)
            # if output <= 0.5 -> predict 0 (normal)
            # Use .view(-1) to ensure predicted is always a 1D tensor
            predicted = (outputs_model.view(-1) > 0.5).float()

            # calc loss
            loss = criterion(outputs_model, y_batch.unsqueeze(1)) # Use outputs_model directly for loss
            test_loss += loss.item()

            #calc accuracy
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

            # collect for f1 score
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    avg_test_loss = test_loss / len(test_loader)
    test_losses.append(avg_test_loss)

    accuracy = correct / total
    test_accuracies.append(accuracy)

    macro_avg = f1_score(all_labels, all_preds, average='macro')
    macro_avgs.append(macro_avg)

    # abnormal recall tracking
    abnormal_mask = np.array(all_labels) == 1
    abnormal_preds = np.array(all_preds)[abnormal_mask]
    abnormal_recall = np.sum(abnormal_preds == 1) / len(abnormal_preds) if len(abnormal_preds) > 0 else 0

    print(f"Epoch {epoch+1}/{num_epochs} - Val Loss: {avg_test_loss:.4f} | Val Acc: {accuracy:.4f} | Macro Avg: {macro_avg:.4f} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}", end="")

    if abnormal_recall > best_abnormal_recall:
        best_abnormal_recall = abnormal_recall
        best_abnormal_recall_state = copy.deepcopy(model.state_dict())
        print(" [new best abnormal recall!]", end="")

    if macro_avg > best_macro_avg:
        best_macro_avg = macro_avg
        best_model_state = copy.deepcopy(model.state_dict())
        patience = 0
        print("\n")
    else:
        patience += 1
        print(f" [{patience}/{max_patience}]\n")

    if patience >= max_patience:
        break

    if (epoch + 1) % 5 == 0:
        print(classification_report(all_labels, all_preds, target_names=['Normal', 'Abnormal'], digits=4))

print(f"Best Macro-Average: {best_macro_avg:.4f}")
model.load_state_dict(best_model_state)

Epoch 1/300 - Val Loss: 0.7069 | Val Acc: 0.4304 | Macro Avg: 0.3491 | Train Loss: 0.6772 | Train Acc: 0.5550 [new best abnormal recall!]

Epoch 2/300 - Val Loss: 0.6921 | Val Acc: 0.4884 | Macro Avg: 0.4674 | Train Loss: 0.6812 | Train Acc: 0.5403

Epoch 3/300 - Val Loss: 0.6881 | Val Acc: 0.5192 | Macro Avg: 0.5171 | Train Loss: 0.6750 | Train Acc: 0.5613

Epoch 4/300 - Val Loss: 0.6510 | Val Acc: 0.6452 | Macro Avg: 0.6452 | Train Loss: 0.6650 | Train Acc: 0.5819

Epoch 5/300 - Val Loss: 0.5228 | Val Acc: 0.7385 | Macro Avg: 0.7382 | Train Loss: 0.5837 | Train Acc: 0.7030

              precision    recall  f1-score   support

      Normal     0.8758    0.6493    0.7457     29099
    Abnormal     0.6315    0.8671    0.7307     20163

    accuracy                         0.7385     49262
   macro avg     0.7536    0.7582    0.7382     49262
weighted avg     0.7758    0.7385    0.7396     49262

Epoch 6/300 - Val Loss: 0.4673 | Val Acc: 0.7715 | Macro Avg: 0.7703 | Train Loss: 0.5234 

KeyboardInterrupt: 

In [ ]:
print(f"Best Macro-Average: {best_macro_avg:.4f}")
model.load_state_dict(best_model_state)

Best Macro-Average: 0.9589


<All keys matched successfully>

In [ ]:
# alter based on where you want your save path
save_path = '/content/drive/MyDrive/'
os.makedirs(save_path, exist_ok=True)

# alter based on your time zone
pst_tz = ZoneInfo("America/Los_Angeles")
today = date.today()
now = datetime.now(tz = pst_tz)
formatted_time = now.strftime("%H:%M")

specific_path = f"{best_macro_avg:.4f}|{str(formatted_time)}|{str(today)}|"
torch.save(model.state_dict(), save_path + specific_path)

training_info = {
    'best_accuracy' : best_accuracy,
    'best_macro_avg' : best_macro_avg,
    'train_losses' : train_losses,
    'test_losses' : test_losses,
    'test_accuracies' : test_accuracies,
    'final_epoch' : len(train_losses)
}

with open(save_path + specific_path + '.pkl', 'wb') as f:
    pickle.dump(training_info, f)

print("model saved to google drive")
print(f"location: {save_path}")
print(f"model file: {specific_path}")
print(f"best macro-avg: {best_macro_avg:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
model saved to google drive
location: /content/drive/MyDrive/ZiyaAhmad/science/Synopsys 2025-2026/Code/best models/normal_vs_abnormal/
model file: 0.9589|17:02|2026-02-19|
best macro-avg: 0.9589
